# TP 3 — Partie 1 : des photos dans le Lakehouse

**Contexte :** Olist ouvre un catalogue photo. Chaque vendeur envoie les photos de ses produits ; une fiche JSON décrit chaque produit (titre, type, marque…). Il faut :
1. **stocker** les photos dans le Lakehouse, à côté des tables ;
2. **contrôler** leur qualité : fichiers illisibles, photos trop petites, doublons ;
3. les **comprendre** : reconnaître automatiquement le type de produit et retrouver les photos qui se ressemblent.

Les photos viennent du jeu public *Amazon Berkeley Objects* (ABO, licence CC BY-NC 4.0) ; des défauts y ont été ajoutés volontairement.

Ce notebook explore et met au point chaque étape. En partie 2, le même code deviendra un **pipeline automatisé**.

> **⚠️ Un seul notebook actif à la fois.** Arrêtez le noyau des autres notebooks avant de commencer.

In [ ]:
import io
import sys

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from pyspark.sql import SparkSession, functions as F

sys.path.insert(0, "pipeline")            # fonctions communes du pipeline (dossier pipeline/)
from commun import analyser, vecteurs

spark = SparkSession.builder.appName("tp3-images").getOrCreate()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


def montrer(liste_octets, legendes, taille=110):
    """Affiche des images côte à côte avec leur légende."""
    images = [Image.open(io.BytesIO(o)).convert("RGB").resize((taille, taille)) for o in liste_octets]
    planche = Image.new("RGB", (taille * len(images), taille), "white")
    for i, img in enumerate(images):
        planche.paste(img, (i * taille, 0))
    display(planche)
    print(" | ".join(legendes))

## Étape 1 — Déposer le premier lot de photos

Le script `deposer_photos.py` simule l'envoi des vendeurs : le **lot 1** (vendeurs 01 à 08) arrive aujourd'hui, avec les fiches produits. Le lot 2 arrivera en partie 2.

In [ ]:
!python /opt/lakehouse/scripts/deposer_photos.py --lot 1

## Étape 2 — Bronze : des fichiers binaires dans une table

Spark lit n'importe quel fichier avec le format `binaryFile` : une ligne par fichier, avec son chemin, sa taille, sa date et son **contenu** (les octets).

Complétez la lecture : format `binaryFile`, option `recursiveFileLookup` à `True` (les photos sont rangées dans un sous-dossier par vendeur), chemin `s3a://landing/catalogue/photos/`. Puis créez `bronze.photos` avec les colonnes : `chemin` (`path`), `vendeur` (extrait du chemin, fourni), `taille_octets` (`length`), `sha256` (`F.sha2("content", 256)` : l'empreinte exacte du fichier), `contenu` (`content`), `_source`, `_ingere_le`.

In [ ]:
brut = a_completer("lecture binaryFile des photos")  # À COMPLÉTER
photos = a_completer("colonnes de bronze.photos")    # À COMPLÉTER
photos.writeTo("lakehouse.bronze.photos").using("iceberg").createOrReplace()
spark.table("lakehouse.bronze.photos").groupBy("vendeur").count().orderBy("vendeur").show()

Les fiches produits (un objet JSON par ligne) entrent en Bronze telles quelles, comme le suivi des colis au TP 2 :

In [ ]:
(spark.read.text("s3a://landing/catalogue/produits.jsonl")
    .withColumn("_source", F.col("_metadata.file_path"))
    .withColumn("_ingere_le", F.current_timestamp())
    .writeTo("lakehouse.bronze.produits_brut").using("iceberg").createOrReplace())
produits = (spark.table("lakehouse.bronze.produits_brut")
            .select(F.from_json("value", "item_id string, type_produit string, titre string, marque string, "
                                         "couleur string, vendeur string, photo string").alias("p"))
            .select("p.*"))
produits.groupBy("type_produit").count().show()

> **❓ Question 1.** Les photos sont stockées **dans** une table Iceberg (colonne `contenu`, de type binaire). Avantages ? Pour des vidéos de plusieurs centaines de Mo, que feriez-vous plutôt ?

## Étape 3 — Silver : décoder et contrôler chaque photo

La fonction `analyser(octets)` (fichier `pipeline/commun.py`) décode une image et renvoie : `valide`, `format`, `largeur` et `hauteur`. On ramène les photos dans pandas (environ 20 Mo) pour les analyser.

In [ ]:
analyse = spark.table("lakehouse.bronze.photos").select("chemin", "vendeur", "sha256", "contenu").toPandas()
analyse[["valide", "format", "largeur", "hauteur"]] = pd.DataFrame(
    [analyser(bytes(o)) for o in analyse.contenu], index=analyse.index)
analyse["item_id"] = analyse.chemin.str.extract(r"/([A-Z0-9]+)(?:_copie|_v2)?\.jpg$", expand=False)
print(analyse.valide.value_counts())
analyse[["chemin", "valide", "format", "largeur", "hauteur", "item_id"]].head()

Regardez les défauts : des fichiers illisibles, des vignettes minuscules, et des produits qui ont **plusieurs photos** : la même, envoyée deux fois (parfois par deux vendeurs), ou une version recadrée.

In [ ]:
print("illisibles :", (~analyse.valide).sum(), "| exemple :", analyse.loc[~analyse.valide, "chemin"].iloc[0][-30:])
print("trop petites :", (analyse.valide & (analyse[["largeur", "hauteur"]].min(axis=1) < 64)).sum())

plusieurs = analyse[analyse.valide & analyse.duplicated("item_id", keep=False)].sort_values("item_id")
print("produits avec plusieurs photos :", plusieurs.item_id.nunique())
for item in [plusieurs[plusieurs.chemin.str.endswith("_copie.jpg")].item_id.iloc[0],
             plusieurs[plusieurs.chemin.str.endswith("_v2.jpg")].item_id.iloc[0]]:
    paire = plusieurs[plusieurs.item_id == item]
    montrer(list(paire.contenu), [f"{c[-34:]} ({s[:8]}…)" for c, s in zip(paire.chemin, paire.sha256)])

Complétez la fonction `raison_de_rejet` : elle reçoit une photo, les empreintes `sha256` et les produits déjà acceptés, et renvoie la raison du rejet ou `None` si la photo est acceptée. Dans l'ordre :
1. `"illisible"` si la photo n'est pas valide ;
2. `"trop_petite"` si sa plus petite dimension est inférieure à 64 pixels ;
3. `"doublon_exact"` si son `sha256` a déjà été accepté (le même fichier, peut-être sous un autre nom) ;
4. `"photo_en_double"` si le produit (`item_id`) a déjà une photo acceptée.

Les photos sont examinées produit par produit, la photo d'origine (au nom le plus court) en premier.

In [ ]:
def raison_de_rejet(p, sha_vus, produits_vus):
    return a_completer("règles de qualité")  # À COMPLÉTER


sha_vus, produits_vus, raisons = set(), set(), []
ordre = analyse.assign(_l=analyse.chemin.str.len()).sort_values(["item_id", "_l"])
for p in ordre.itertuples():
    r = raison_de_rejet(p, sha_vus, produits_vus)
    if r is None:
        sha_vus.add(p.sha256)
        produits_vus.add(p.item_id)
    raisons.append((p.chemin, r))
analyse = analyse.merge(pd.DataFrame(raisons, columns=["chemin", "raison"]), on="chemin")
print(analyse.raison.fillna("acceptée").value_counts())

Écriture des tables Silver : les photos acceptées d'un côté, la quarantaine de l'autre.

In [ ]:
colonnes = ["chemin", "vendeur", "item_id", "sha256", "format", "largeur", "hauteur"]
sortie = analyse.copy()
for c in ["largeur", "hauteur"]:
    sortie[c] = sortie[c].astype("Int64")
sortie = sortie.astype(object).where(sortie.notna(), None)
schema = "chemin string, vendeur string, item_id string, sha256 string, format string, largeur int, hauteur int"
(spark.createDataFrame(sortie.loc[sortie.raison.isna(), colonnes], schema)
    .writeTo("lakehouse.silver.photos").using("iceberg").createOrReplace())
(spark.createDataFrame(sortie.loc[sortie.raison.notna(), colonnes + ["raison"]], schema + ", raison string")
    .writeTo("lakehouse.silver.quarantaine_photos").using("iceberg").createOrReplace())
print("silver.photos :", spark.table("lakehouse.silver.photos").count(),
      "| quarantaine :", spark.table("lakehouse.silver.quarantaine_photos").count())

> **❓ Question 2.** Pourquoi deux contrôles de doublons ? Que rate l'empreinte `sha256` ? Pourquoi ne pas comparer les photos « à l'œil », d'un produit à l'autre, pour trouver les doublons ?

## Étape 4 — Gold : ce que « voit » un réseau de neurones

Un réseau de neurones pré-entraîné sur des millions d'images (MobileNet, fichier `data/modeles/mobilenet_v3_small.onnx`) transforme chaque photo en un **vecteur de 576 nombres** qui résume son contenu visuel : formes, textures, couleurs. Deux photos d'objets semblables ont des vecteurs proches. La fonction `vecteurs(liste_octets)` est fournie ; elle prend environ une minute.

In [ ]:
acceptees = analyse[analyse.raison.isna()].reset_index(drop=True)
V = vecteurs([bytes(o) for o in acceptees.contenu])
print("vecteurs :", V.shape)

fiches = produits.toPandas()
acceptees = acceptees.merge(fiches[["item_id", "type_produit", "titre"]], on="item_id", how="left")
(spark.createDataFrame(pd.DataFrame({"chemin": acceptees.chemin, "item_id": acceptees.item_id,
                                     "vecteur": [list(map(float, v)) for v in V]}),
                       "chemin string, item_id string, vecteur array<float>")
    .writeTo("lakehouse.gold.photos_vecteurs").using("iceberg").createOrReplace())

**Recherche d'images similaires.** La similarité cosinus mesure si deux vecteurs pointent dans la même direction (1 = identiques). Complétez `similaires(i, n)` : elle renvoie les indices des `n` photos les plus proches de la photo `i` (sans la photo elle-même).

In [ ]:
Vn = V / np.linalg.norm(V, axis=1, keepdims=True)     # vecteurs de longueur 1 : produit scalaire = cosinus


def similaires(i, n=5):
    return a_completer("indices des n photos les plus proches")  # À COMPLÉTER


for i in [0, 700, 1400]:
    voisins = [i] + similaires(i)
    montrer([bytes(acceptees.contenu[j]) for j in voisins],
            [acceptees.type_produit[j] for j in voisins])

**Des fiches différentes aux photos identiques.** Cherchons les paires de photos de produits **différents** dont la similarité dépasse 0,97 : ce sont souvent des variantes d'un même modèle (une autre couleur, une autre taille), parfois une fiche créée deux fois. On ne les rejette pas : on les signale à l'équipe catalogue.

In [ ]:
S = Vn @ Vn.T
np.fill_diagonal(S, 0)
paires = np.argwhere(np.triu(S) >= 0.97)
print(len(paires), "paires de fiches différentes aux photos quasi identiques")
for a, b in paires[:3]:
    montrer([bytes(acceptees.contenu[a]), bytes(acceptees.contenu[b])], [f"{S[a, b]:.3f}", ""])
    print("   ", acceptees.titre[a][:70], "\n   ", acceptees.titre[b][:70])

**Reconnaître le type de produit.** Une régression logistique, entraînée sur les vecteurs, apprend à prédire le `type_produit` déclaré dans la fiche. Les prédictions sont faites « hors échantillon » (`cross_val_predict`) : chaque photo est classée par un modèle qui ne l'a pas vue.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_predict

y = acceptees.type_produit
proba = cross_val_predict(LogisticRegression(max_iter=2000, C=0.5), V, y, cv=5, method="predict_proba")
classes = np.array(sorted(y.unique()))
acceptees["type_predit"] = classes[proba.argmax(axis=1)]
acceptees["confiance"] = proba.max(axis=1).round(3)
print(classification_report(y, acceptees.type_predit))

Les fiches dont la photo contredit le type déclaré, avec une forte confiance, sont signalées : ce sont des **erreurs de catégorie** probables, à vérifier par l'équipe catalogue.

In [ ]:
acceptees["incoherent"] = (acceptees.type_predit != acceptees.type_produit) & (acceptees.confiance >= 0.8)
douteuses = acceptees[acceptees.incoherent].sort_values("confiance", ascending=False).head(6)
montrer([bytes(o) for o in douteuses.contenu],
        [f"{a} → {b}" for a, b in zip(douteuses.type_produit, douteuses.type_predit)])
print(douteuses[["titre", "type_produit", "type_predit", "confiance"]].to_string())

(spark.createDataFrame(acceptees[["item_id", "chemin", "titre", "type_produit", "type_predit", "confiance", "incoherent"]])
    .writeTo("lakehouse.gold.photos_controle").using("iceberg").createOrReplace())

> **❓ Question 3.** Lisez les titres des fiches signalées. Le modèle se trompe-t-il, ou la fiche est-elle mal renseignée ? Le modèle n'a jamais vu de « chaise » pendant son entraînement d'origine : comment peut-il les reconnaître ?

## Bilan

In [ ]:
for t in ["bronze.photos", "bronze.produits_brut", "silver.photos", "silver.quarantaine_photos",
          "gold.photos_vecteurs", "gold.photos_controle"]:
    print(f"{t:28s}", spark.table(f"lakehouse.{t}").count())
spark.table("lakehouse.silver.quarantaine_photos").groupBy("raison").count().show()